In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Advanced Pyspark").getOrCreate()

In [3]:
from google.colab import files

uploaded = files.upload()

Saving machine_configurations.json to machine_configurations.json
Saving plant_master.csv to plant_master.csv
Saving manufacturing_production_extra_12k.csv to manufacturing_production_extra_12k.csv
Saving manufacturing_production_120k.csv to manufacturing_production_120k.csv
Saving machine_master.csv to machine_master.csv


**1. Load the Data in PySpark**

In [4]:
df = spark.read.csv(
"manufacturing_production_120k.csv",
    header=True,
    inferSchema=True
)

1. Display first 20 rows.

In [ ]:
df.show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Evening|     MTR-100|        Motors|         1060|           993|             30|              34|    388.11|         61.4|          3.88|        Rework|              No|    Charan|
|    PR0000002|     

2. Display schema.

In [ ]:
df.printSchema()

root
 |-- production_id: string (nullable = true)
 |-- production_date: date (nullable = true)
 |-- plant_id: string (nullable = true)
 |-- machine_id: string (nullable = true)
 |-- operator_id: string (nullable = true)
 |-- shift: string (nullable = true)
 |-- product_code: string (nullable = true)
 |-- product_family: string (nullable = true)
 |-- units_planned: integer (nullable = true)
 |-- units_produced: integer (nullable = true)
 |-- defective_units: integer (nullable = true)
 |-- downtime_minutes: integer (nullable = true)
 |-- energy_kwh: double (nullable = true)
 |-- temperature_c: double (nullable = true)
 |-- vibration_mm_s: double (nullable = true)
 |-- quality_status: string (nullable = true)
 |-- maintenance_flag: string (nullable = true)
 |-- supervisor: string (nullable = true)



3. Display column names.

In [ ]:
print(df.columns)

['production_id', 'production_date', 'plant_id', 'machine_id', 'operator_id', 'shift', 'product_code', 'product_family', 'units_planned', 'units_produced', 'defective_units', 'downtime_minutes', 'energy_kwh', 'temperature_c', 'vibration_mm_s', 'quality_status', 'maintenance_flag', 'supervisor']


4. Count total records.

In [ ]:
print(df.count())

120000


5. Display only production ID, plant, machine, product family and units produced.

In [ ]:
df.select("production_id", "plant_id", "machine_id", "product_family", "units_produced").show()

+-------------+--------+----------+--------------+--------------+
|production_id|plant_id|machine_id|product_family|units_produced|
+-------------+--------+----------+--------------+--------------+
|    PR0000001|    PL03|     MC011|        Motors|           993|
|    PR0000002|    PL04|     MC019|   Controllers|           836|
|    PR0000003|    PL01|     MC002|        Motors|           380|
|    PR0000004|    PL03|     MC013|         Pumps|           720|
|    PR0000005|    PL04|     MC020|         Pumps|           296|
|    PR0000006|    PL01|     MC002|        Motors|           795|
|    PR0000007|    PL02|     MC007|        Valves|           955|
|    PR0000008|    PL02|     MC009|        Valves|           895|
|    PR0000009|    PL03|     MC015|   Controllers|           289|
|    PR0000010|    PL02|     MC006|   Controllers|           527|
|    PR0000011|    PL03|     MC012|         Pumps|           512|
|    PR0000012|    PL06|     MC030|   Controllers|           968|
|    PR000

6. Check the datatype of each column.

In [ ]:
print(df.dtypes)

[('production_id', 'string'), ('production_date', 'date'), ('plant_id', 'string'), ('machine_id', 'string'), ('operator_id', 'string'), ('shift', 'string'), ('product_code', 'string'), ('product_family', 'string'), ('units_planned', 'int'), ('units_produced', 'int'), ('defective_units', 'int'), ('downtime_minutes', 'int'), ('energy_kwh', 'double'), ('temperature_c', 'double'), ('vibration_mm_s', 'double'), ('quality_status', 'string'), ('maintenance_flag', 'string'), ('supervisor', 'string')]


7. Display 10 random-looking sample records using sample().

In [5]:
df.sample(withReplacement=False, fraction=0.1, seed=42).show(10)

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0000008|     2026-08-17|    PL02|     MC009|     OP0106|  Night|     VLV-320|        Valves|          862|           895|             51|              77|    266.06|         57.1|          1.34|        Reject|              No|     Irfan|
|    PR0000017|     

**2. Creating DataFrames**

Create a small DataFrame manually containing: plant_id, plant_name, city

In [ ]:
plant_data = [(1, "Plant Alpha", "Hyderabad"),
              (2, "Plant Beta", "Chennai"),
              (3, "Plant Gamma", "Pune")]

1. Create it using a list of tuples.

In [ ]:
small_df = spark.createDataFrame(plant_data)

2. Create it with explicit column names.

In [ ]:
small_df = spark.createDataFrame(plant_data, ["plant_id", "plant_name", "city"])

3. Create another version using StructType.

In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

schema = StructType([
    StructField("plant_id", IntegerType(), True),
    StructField("plant_name", StringType(), True),
    StructField("city", StringType(), True)
])

4. Print the schema.

In [ ]:
small_df_st = spark.createDataFrame(plant_data, schema)
small_df_st.printSchema()

root
 |-- plant_id: integer (nullable = true)
 |-- plant_name: string (nullable = true)
 |-- city: string (nullable = true)



5. Add a constant column called country = India.

In [ ]:
from pyspark.sql.functions import lit
small_df_st = small_df_st.withColumn("country", lit("India")).show()

+--------+-----------+---------+-------+
|plant_id| plant_name|     city|country|
+--------+-----------+---------+-------+
|       1|Plant Alpha|Hyderabad|  India|
|       2| Plant Beta|  Chennai|  India|
|       3|Plant Gamma|     Pune|  India|
+--------+-----------+---------+-------+



**3. Pandas DataFrame.**

In [ ]:
import pandas as pd
pdf = pd.read_csv("manufacturing_production_120k.csv")

1. Display first 5 rows.

In [ ]:
print(pdf.head())

  production_id production_date plant_id machine_id operator_id    shift  \
0     PR0000001      2026-08-07     PL03      MC011      OP0275  Evening   
1     PR0000002      2026-08-03     PL04      MC019      OP0489  Morning   
2     PR0000003      2026-05-20     PL01      MC002      OP0189  Evening   
3     PR0000004      2026-01-15     PL03      MC013      OP0076  Morning   
4     PR0000005      2026-09-21     PL04      MC020      OP0501  Morning   

  product_code product_family  units_planned  units_produced  defective_units  \
0      MTR-100         Motors           1060             993               30   
1      CTL-330    Controllers            809             836                0   
2      MTR-300         Motors            442             380               17   
3      PMP-210          Pumps            787             720                7   
4      PMP-110          Pumps            344             296                7   

   downtime_minutes  energy_kwh  temperature_c  vibratio

2. Display last 5 rows.

In [ ]:
print(pdf.tail())

       production_id production_date plant_id machine_id operator_id    shift  \
119995     PR0119996      2026-07-13     PL05      MC022      OP0193  Evening   
119996     PR0119997      2026-04-26     PL05      MC024      OP0124  Evening   
119997     PR0119998      2026-02-01     PL06      MC027      OP0051  Evening   
119998     PR0119999      2026-01-17     PL01      MC001      OP0452  Evening   
119999     PR0120000      2026-08-16     PL02      MC007      OP0420  Morning   

       product_code product_family  units_planned  units_produced  \
119995      MTR-200         Motors            435             399   
119996      CTL-130    Controllers            813             777   
119997      VLV-120         Valves            650             597   
119998      MTR-300         Motors            887             899   
119999      MTR-200         Motors            547             477   

        defective_units  downtime_minutes  energy_kwh  temperature_c  \
119995                1   

3. Display shape.

In [ ]:
print(pdf.shape)

(120000, 18)


4. Display column names.

In [ ]:
print(pdf.columns)

Index(['production_id', 'production_date', 'plant_id', 'machine_id',
       'operator_id', 'shift', 'product_code', 'product_family',
       'units_planned', 'units_produced', 'defective_units',
       'downtime_minutes', 'energy_kwh', 'temperature_c', 'vibration_mm_s',
       'quality_status', 'maintenance_flag', 'supervisor'],
      dtype='object')


5. Display datatypes.

In [ ]:
print(pdf.dtypes)

production_id        object
production_date      object
plant_id             object
machine_id           object
operator_id          object
shift                object
product_code         object
product_family       object
units_planned         int64
units_produced        int64
defective_units       int64
downtime_minutes    float64
energy_kwh          float64
temperature_c       float64
vibration_mm_s      float64
quality_status       object
maintenance_flag     object
supervisor           object
dtype: object


6. Select plant_id , machine_id , units_produced.

In [ ]:
pdf[["plant_id", "machine_id", "units_produced"]].head()

,plant_id,machine_id,units_produced
0,PL03,MC011,993
1,PL04,MC019,836
2,PL01,MC002,380
3,PL03,MC013,720
4,PL04,MC020,296


7. Rename units_produced to production_output.

In [ ]:
pdf_renamed = pdf.rename(columns={"units_produced": "production_output"})
pdf_renamed.head()

,production_id,production_date,plant_id,machine_id,operator_id,shift,product_code,product_family,units_planned,production_output,defective_units,downtime_minutes,energy_kwh,temperature_c,vibration_mm_s,quality_status,maintenance_flag,supervisor
0,PR0000001,2026-08-07,PL03,MC011,OP0275,Evening,MTR-100,Motors,1060,993,30,34.0,388.11,61.40,3.88,Rework,No,Charan
1,PR0000002,2026-08-03,PL04,MC019,OP0489,Morning,CTL-330,Controllers,809,836,0,0.0,398.58,64.37,2.28,Reject,No,Farah
2,PR0000003,2026-05-20,PL01,MC002,OP0189,Evening,MTR-300,Motors,442,380,17,15.0,183.34,54.13,4.66,Reject,No,Deepa
3,PR0000004,2026-01-15,PL03,MC013,OP0076,Morning,PMP-210,Pumps,787,720,7,39.0,217.61,64.47,NaN,Rework,No,Farah
4,PR0000005,2026-09-21,PL04,MC020,OP0501,Morning,PMP-110,Pumps,344,296,7,49.0,115.47,72.59,3.21,Pass,No,Deepa


8. Filter rows where quality_status == "Reject".

In [ ]:
print(pdf.filter(pdf["quality_status"] == "Reject").head())

Empty DataFrame
Columns: []
Index: [0, 1, 2, 3, 4]


9. Filter rows where temperature is greater than 75.

In [ ]:
print(pdf.filter(pdf["temperature_c"] > 75).head())

Empty DataFrame
Columns: []
Index: [0, 1, 2, 3, 4]


10. Filter Night-shift production from Hyderabad plant.

In [ ]:
plant_pdf = pd.read_csv("plant_master.csv")
hyderabad_plant = plant_pdf[plant_pdf["city"] == "Hyderabad"]
print(pdf[(pdf["shift"] == "Night") & (pdf["plant_id"].isin(hyderabad_plant["plant_id"]))].head())

    production_id production_date plant_id machine_id operator_id  shift  \
29      PR0000030      2026-09-18     PL01      MC004      OP0266  Night   
38      PR0000039      2026-07-17     PL01      MC002      OP0220  Night   
62      PR0000063      2026-02-18     PL01      MC002      OP0390  Night   
79      PR0000080      2026-06-10     PL01      MC003      OP0214  Night   
145     PR0000146      2026-08-19     PL01      MC005      OP0088  Night   

    product_code product_family  units_planned  units_produced  \
29       VLV-320         Valves            358             359   
38       VLV-120         Valves            260             265   
62       VLV-120         Valves            656             665   
79       CTL-130    Controllers            948             821   
145      VLV-120         Valves           1156             969   

     defective_units  downtime_minutes  energy_kwh  temperature_c  \
29                 1              19.0      107.98          72.00   
38      

**4. Dask DataFrame**

In [ ]:
import dask.dataframe as dd
import time
ddf = dd.read_csv("manufacturing_production_120k.csv")

1. Display first records using head().

In [ ]:
ddf = dd.read_csv("manufacturing_production_120k.csv", dtype={'downtime_minutes': 'float64'})
print(ddf.head())

  production_id production_date plant_id machine_id operator_id    shift  \
0     PR0000001      2026-08-07     PL03      MC011      OP0275  Evening   
1     PR0000002      2026-08-03     PL04      MC019      OP0489  Morning   
2     PR0000003      2026-05-20     PL01      MC002      OP0189  Evening   
3     PR0000004      2026-01-15     PL03      MC013      OP0076  Morning   
4     PR0000005      2026-09-21     PL04      MC020      OP0501  Morning   

  product_code product_family  units_planned  units_produced  defective_units  \
0      MTR-100         Motors           1060             993               30   
1      CTL-330    Controllers            809             836                0   
2      MTR-300         Motors            442             380               17   
3      PMP-210          Pumps            787             720                7   
4      PMP-110          Pumps            344             296                7   

   downtime_minutes  energy_kwh  temperature_c  vibratio

2. Find number of partitions.


In [ ]:
print(ddf.npartitions)

1


3. Select plant_id , product_family , units_produced.

In [ ]:
selected_columns = ddf[["plant_id", "product_family", "units_produced"]]
print(selected_columns.head())


  plant_id product_family  units_produced
0     PL03         Motors             993
1     PL04    Controllers             836
2     PL01         Motors             380
3     PL03          Pumps             720
4     PL04          Pumps             296


4. Filter rejected production records.

In [ ]:
rejected_records = ddf[ddf["quality_status"] == "Reject"]
print(rejected_records.head())

   production_id production_date plant_id machine_id operator_id    shift  \
1      PR0000002      2026-08-03     PL04      MC019      OP0489  Morning   
2      PR0000003      2026-05-20     PL01      MC002      OP0189  Evening   
7      PR0000008      2026-08-17     PL02      MC009      OP0106    Night   
9      PR0000010      2026-02-13     PL02      MC006      OP0593  Evening   
21     PR0000022      2026-03-03     PL02      MC009      OP0294  Evening   

   product_code product_family  units_planned  units_produced  \
1       CTL-330    Controllers            809             836   
2       MTR-300         Motors            442             380   
7       VLV-320         Valves            862             895   
9       CTL-230    Controllers            532             527   
21      MTR-300         Motors            623             560   

    defective_units  downtime_minutes  energy_kwh  temperature_c  \
1                 0               0.0      398.58          64.37   
2         

5. Calculate average units produced by plant.

In [ ]:
average_units_by_plant = ddf.groupby("plant_id")["units_produced"].mean().compute()
print(average_units_by_plant)

plant_id
PL01    672.998304
PL02    672.781326
PL03    675.665503
PL04    673.835101
PL05    673.124994
PL06    678.006884
Name: units_produced, dtype: float64


6. Calculate total defects by product family.

In [ ]:
defects_by_product_family = ddf.groupby("product_family")["defective_units"].sum().compute()
print(defects_by_product_family)

product_family
Controllers    507781
Motors         505287
Pumps          507864
Valves         509312
Name: defective_units, dtype: int64


7. Use .compute() to obtain the final result.

In [ ]:
# The result is already computed and is a Pandas Series, so we print it directly
print(average_units_by_plant)
print(defects_by_product_family)

plant_id
PL01    672.998304
PL02    672.781326
PL03    675.665503
PL04    673.835101
PL05    673.124994
PL06    678.006884
Name: units_produced, dtype: float64
product_family
Controllers    507781
Motors         505287
Pumps          507864
Valves         509312
Name: defective_units, dtype: int64


8. Compare doing the same aggregation in Pandas and Dask.

In [ ]:
t0 = time.time(); p_res = pdf.groupby("plant_id")["units_produced"].mean(); t_pd = time.time() - t0
t0 = time.time(); d_res = ddf.groupby("plant_id")["units_produced"].mean().compute(); t_dd = time.time() - t0
print(f"Pandas: {t_pd:.3f}s | Dask: {t_dd:.3f}s")

Pandas: 0.015s | Dask: 0.388s


**5. Manipulating DataFrames**

In [ ]:
df = spark.read.csv(
"manufacturing_production_120k.csv",
    header=True,
    inferSchema=True
)

In [ ]:
df.show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Evening|     MTR-100|        Motors|         1060|           993|             30|              34|    388.11|         61.4|          3.88|        Rework|              No|    Charan|
|    PR0000002|     

1. Create: production_efficiency =
units_produced / units_planned * 100

In [ ]:
df2 = df.withColumn("production_efficiency", (df["units_produced"] / df["units_planned"]) * 100)
df2.show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+---------------------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|production_efficiency|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+---------------------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Evening|     MTR-100|        Motors|         1060|           993|             30|              34|    388.11|         61.4|          3.8

2. Create: good_units =
units_produced - defective_units

In [ ]:
df2 = df2.withColumn("good_units", df2["units_produced"] - df2["defective_units"])
df2.show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+---------------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|production_efficiency|good_units|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+---------------------+----------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Evening|     MTR-100|        Motors|         1060|           993|             30|              34|    3

3. Add: country = India

In [ ]:
df2 = df2.withColumn("country", lit("India"))
df2.show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+---------------------+----------+-------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|production_efficiency|good_units|country|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+---------------------+----------+-------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Evening|     MTR-100|        Motors|         1060|           993|             3

4. Rename energy_kwh to energy_consumed

In [ ]:
df2 = df2.withColumnRenamed("energy_kwh", "energy_consumed")
df2.show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+---------------+-------------+--------------+--------------+----------------+----------+---------------------+----------+-------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_consumed|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|production_efficiency|good_units|country|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+---------------+-------------+--------------+--------------+----------------+----------+---------------------+----------+-------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Evening|     MTR-100|        Motors|         1060|           993

5. Convert units_produced to double.

In [ ]:
df2 = df2.withColumn("units_produced", df2["units_produced"].cast("double"))
df2.show()
print(df2.dtypes)

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+---------------+-------------+--------------+--------------+----------------+----------+---------------------+----------+-------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_consumed|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|production_efficiency|good_units|country|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+---------------+-------------+--------------+--------------+----------------+----------+---------------------+----------+-------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Evening|     MTR-100|        Motors|         1060|         993.0

6. Create defect_percentage.

In [ ]:
df2 = df2.withColumn("defect_percentage", (df2["defective_units"] / df2["units_produced"]) * 100)
df2.show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+---------------+-------------+--------------+--------------+----------------+----------+---------------------+----------+-------+------------------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_consumed|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|production_efficiency|good_units|country| defect_percentage|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+---------------+-------------+--------------+--------------+----------------+----------+---------------------+----------+-------+------------------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Evening

7. Create energy_per_unit.

In [ ]:
df2 = df2.withColumn("energy_per_unit", df2["energy_consumed"] / df2["units_produced"])
df2.show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+---------------+-------------+--------------+--------------+----------------+----------+---------------------+----------+-------+------------------+-------------------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_consumed|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|production_efficiency|good_units|country| defect_percentage|    energy_per_unit|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+---------------+-------------+--------------+--------------+----------------+----------+---------------------+----------+-------+------------------+-------------------+
|    PR000

8. Select only the transformed columns.

In [ ]:
df2.select("production_efficiency", "good_units", "country", "energy_consumed", "defect_percentage", "energy_per_unit").show()

+---------------------+----------+-------+---------------+------------------+-------------------+
|production_efficiency|good_units|country|energy_consumed| defect_percentage|    energy_per_unit|
+---------------------+----------+-------+---------------+------------------+-------------------+
|    93.67924528301886|       963|  India|         388.11|3.0211480362537766|0.39084592145015107|
|   103.33745364647713|       836|  India|         398.58|               0.0| 0.4767703349282296|
|    85.97285067873304|       363|  India|         183.34| 4.473684210526316|0.48247368421052633|
|     91.4866581956798|       713|  India|         217.61|0.9722222222222222|0.30223611111111115|
|    86.04651162790698|       289|  India|         115.47| 2.364864864864865| 0.3901013513513513|
|    82.98538622129436|       745|  India|         290.26| 6.289308176100629| 0.3651069182389937|
|    81.97424892703863|       945|  India|         355.56|1.0471204188481675|0.37231413612565445|
|   103.828306264501

**6. Dropping Data**


1. Drop supervisor.

In [ ]:
df.drop("supervisor").show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Evening|     MTR-100|        Motors|         1060|           993|             30|              34|    388.11|         61.4|          3.88|        Rework|              No|
|    PR0000002|     2026-08-03|    PL04|     MC019|     OP0489|M

2. Drop supervisor and maintenance_flag.

In [ ]:
df.drop("supervisor", "maintenance_flag").show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Evening|     MTR-100|        Motors|         1060|           993|             30|              34|    388.11|         61.4|          3.88|        Rework|
|    PR0000002|     2026-08-03|    PL04|     MC019|     OP0489|Morning|     CTL-330|   Controllers|          809|           836|    

3. Remove duplicate rows.

In [ ]:
df.dropDuplicates().count()

120000

4. Keep one record per production_id.

In [ ]:
df.dropDuplicates(["production_id"]).show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0000002|     2026-08-03|    PL04|     MC019|     OP0489|Morning|     CTL-330|   Controllers|          809|           836|              0|               0|    398.58|        64.37|          2.28|        Reject|              No|     Farah|
|    PR0000003|     

5. Remove records where operator ID is missing.

In [ ]:
df.na.drop(subset=["operator_id"]).show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Evening|     MTR-100|        Motors|         1060|           993|             30|              34|    388.11|         61.4|          3.88|        Rework|              No|    Charan|
|    PR0000002|     

6. Remove rows where both temperature and vibration are missing.

In [ ]:
df.na.drop(subset=["temperature_c", "vibration_mm_s"], how="all").show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Evening|     MTR-100|        Motors|         1060|           993|             30|              34|    388.11|         61.4|          3.88|        Rework|              No|    Charan|
|    PR0000002|     

7. Display unique product families.

In [ ]:
df.select("product_family").distinct().show()

+--------------+
|product_family|
+--------------+
|         Pumps|
|        Valves|
|   Controllers|
|        Motors|
+--------------+



8. Display unique plant-machine combinations.

In [ ]:
df.select("plant_id", "machine_id").distinct().show()

+--------+----------+
|plant_id|machine_id|
+--------+----------+
|    PL01|     MC001|
|    PL01|     MC004|
|    PL04|     MC019|
|    PL04|     MC017|
|    PL05|     MC022|
|    PL04|     MC018|
|    PL01|     MC003|
|    PL06|     MC029|
|    PL02|     MC007|
|    PL03|     MC013|
|    PL06|     MC030|
|    PL04|     MC020|
|    PL05|     MC021|
|    PL01|     MC005|
|    PL02|     MC010|
|    PL04|     MC016|
|    PL06|     MC027|
|    PL02|     MC006|
|    PL01|     MC002|
|    PL06|     MC028|
+--------+----------+
only showing top 20 rows


**7. Sorting**

In [ ]:
from pyspark.sql.functions import asc, desc

1. Sort by units produced ascending.

In [ ]:
from pyspark.sql.functions import col
df.sort(col("units_produced").asc()).show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0038643|     2026-08-06|    PL02|     MC009|     OP0361|Evening|     MTR-100|        Motors|          250|           205|              6|              73|     65.18|        55.99|          2.72|          Pass|              No|    Harini|
|    PR0059668|     

2. Sort units produced descending.

In [ ]:
df.sort(col("units_produced").desc()).show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0087560|     2026-06-29|    PL04|     MC018|     OP0587|  Night|     VLV-120|        Valves|         1199|          1245|             60|               0|    500.87|        58.11|          3.58|          Pass|             Yes|    Charan|
|    PR0008535|     

3. Sort highest defective units first.

In [ ]:
df.sort(col("defective_units").desc()).show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0051359|     2026-09-16|    PL02|     MC009|     OP0181|Morning|     CTL-230|   Controllers|         1192|          1230|            108|               8|    555.34|        71.97|          2.09|        Rework|              No|     Irfan|
|    PR0108009|     

4. Sort by plant and then production output descending.

In [ ]:
df.sort(col("plant_id").asc(), col("units_produced").desc()).show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0073059|     2026-04-01|    PL01|     MC003|     OP0110|Evening|     MTR-200|        Motors|         1198|          1239|             11|               0|     282.6|        50.32|          3.91|        Rework|              No|     Gopal|
|    PR0002678|     

5. Display the top 20 production runs.

In [ ]:
df.sort(col("units_produced").desc()).limit(20).show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0087560|     2026-06-29|    PL04|     MC018|     OP0587|  Night|     VLV-120|        Valves|         1199|          1245|             60|               0|    500.87|        58.11|          3.58|          Pass|             Yes|    Charan|
|    PR0008535|     

6. Display the 10 runs with highest downtime.

In [ ]:
df.sort(col("downtime_minutes").desc()).limit(10).show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0092818|     2026-02-16|    PL06|     MC027|     OP0016|Morning|     PMP-210|         Pumps|         1066|           907|              2|             281|    359.48|        54.03|           2.8|          Pass|              No|     Farah|
|    PR0045363|     

7. Find machines with the highest recorded vibration.

In [ ]:
df.sort(col("vibration_mm_s").desc()).limit(10).show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0020211|     2026-10-08|    PL03|     MC012|     OP0283|  Night|     MTR-200|        Motors|          797|           820|             32|               0|    310.44|        54.91|          8.56|        Rework|             Yes|     Farah|
|    PR0109182|     

**8. Aggregations**

In [ ]:
from pyspark.sql.functions import sum, avg, min, max, count

1. Total units planned.

In [ ]:
df.agg(sum("units_planned").alias("total_units_planned")).show()

+-------------------+
|total_units_planned|
+-------------------+
|           87094621|
+-------------------+



2. Total units produced.

In [ ]:
df.agg(sum("units_produced").alias("total_units_produced")).show()

+--------------------+
|total_units_produced|
+--------------------+
|            80927888|
+--------------------+



3. Total defective units.

In [ ]:
df.agg(sum("defective_units").alias("total_defective_units")).show()

+---------------------+
|total_defective_units|
+---------------------+
|              2030244|
+---------------------+



4. Average units produced.

In [ ]:
df.agg(avg("units_produced").alias("average_units_produced")).show()

+----------------------+
|average_units_produced|
+----------------------+
|     674.3990666666666|
+----------------------+



5. Maximum production output.

In [ ]:
df.agg(max("units_produced").alias("maximum_units_produced")).show()

+----------------------+
|maximum_units_produced|
+----------------------+
|                  1245|
+----------------------+



6. Minimum production output.

In [ ]:
df.agg(min("units_produced").alias("minimum_units_produced")).show()

+----------------------+
|minimum_units_produced|
+----------------------+
|                   205|
+----------------------+



7. Average downtime.

In [ ]:
df.agg(avg("downtime_minutes").alias("average_downtime")).show()

+-----------------+
| average_downtime|
+-----------------+
|33.93149474428351|
+-----------------+



8. Total energy consumed.

In [ ]:
df.agg(sum("energy_kwh").alias("total_energy_consumed")).show()

+---------------------+
|total_energy_consumed|
+---------------------+
| 2.8368014080000117E7|
+---------------------+



9. Average temperature.

In [ ]:
df.agg(avg("temperature_c").alias("average_temperature")).show()

+-------------------+
|average_temperature|
+-------------------+
| 63.021532392167266|
+-------------------+



10. Average vibration.

In [ ]:
df.agg(avg("vibration_mm_s").alias("average_vibration")).show()

+-----------------+
|average_vibration|
+-----------------+
|3.398737691604917|
+-----------------+



**9. groupBy()**


Sort each summary appropriately.

In [ ]:
def summarize(keys, agg_expr, alias):
    keys = [keys] if isinstance(keys, str) else keys
    return df.groupBy(*keys).agg(agg_expr.alias(alias)).orderBy(F.col(alias).desc())

In [ ]:
from pyspark.sql import functions as F

1. Total production by plant.

In [ ]:
summarize("plant_id", F.sum("units_produced"), "total_production").show()

+--------+----------------+
|plant_id|total_production|
+--------+----------------+
|    PL03|        13547769|
|    PL02|        13546452|
|    PL04|        13529935|
|    PL06|        13492337|
|    PL01|        13488232|
|    PL05|        13323163|
+--------+----------------+



2. Total production by product family.

In [ ]:
summarize("product_family", F.sum("units_produced"), "total_production").show()

+--------------+----------------+
|product_family|total_production|
+--------------+----------------+
|        Valves|        20433082|
|         Pumps|        20208739|
|   Controllers|        20168572|
|        Motors|        20117495|
+--------------+----------------+



3. Total defects by plant.

In [ ]:
summarize("plant_id", F.sum("defective_units"), "total_defects").show()

+--------+-------------+
|plant_id|total_defects|
+--------+-------------+
|    PL01|       339720|
|    PL02|       339647|
|    PL03|       339264|
|    PL06|       338993|
|    PL04|       338118|
|    PL05|       334502|
+--------+-------------+



4. Average downtime by shift.

In [ ]:
summarize("shift", F.avg("downtime_minutes"), "average_downtime").show()

+-------+------------------+
|  shift|  average_downtime|
+-------+------------------+
|Evening|  34.2213017151833|
|  Night|33.953058820580935|
|Morning| 33.61973080530278|
+-------+------------------+



5. Average energy usage by product family.

In [ ]:
summarize("product_family", F.avg("energy_kwh"), "average_energy").show()

+--------------+------------------+
|product_family|    average_energy|
+--------------+------------------+
|        Motors|237.21818224597516|
|         Pumps|236.89137927582252|
|        Valves|236.34508750495314|
|   Controllers|235.15370433333288|
+--------------+------------------+



6. Production count by quality status.

In [ ]:
summarize("quality_status", F.count("*"), "production_count").show()

+--------------+----------------+
|quality_status|production_count|
+--------------+----------------+
|          Pass|           74328|
|        Rework|           22928|
|        Reject|           22744|
+--------------+----------------+



7. Production by plant and shift.

In [ ]:
summarize(["plant_id", "shift"], F.sum("units_produced"), "total_production").show()

+--------+-------+----------------+
|plant_id|  shift|total_production|
+--------+-------+----------------+
|    PL06|  Night|         4604432|
|    PL03|  Night|         4599608|
|    PL01|  Night|         4562612|
|    PL02|Morning|         4561068|
|    PL02|  Night|         4558467|
|    PL04|Evening|         4540617|
|    PL04|Morning|         4497099|
|    PL04|  Night|         4492219|
|    PL03|Evening|         4487211|
|    PL01|Evening|         4477769|
|    PL05|Evening|         4475101|
|    PL03|Morning|         4460950|
|    PL06|Evening|         4458798|
|    PL01|Morning|         4447851|
|    PL05|Morning|         4431446|
|    PL06|Morning|         4429107|
|    PL02|Evening|         4426917|
|    PL05|  Night|         4416616|
+--------+-------+----------------+



8. Defects by plant and product family.

In [ ]:
summarize(["plant_id", "product_family"], F.sum("defective_units"), "total_defects").show()

+--------+--------------+-------------+
|plant_id|product_family|total_defects|
+--------+--------------+-------------+
|    PL02|   Controllers|        87120|
|    PL01|         Pumps|        87103|
|    PL04|        Valves|        87084|
|    PL06|   Controllers|        86140|
|    PL06|        Valves|        85570|
|    PL03|        Valves|        85333|
|    PL03|        Motors|        84926|
|    PL01|        Valves|        84874|
|    PL03|         Pumps|        84803|
|    PL02|        Motors|        84560|
|    PL02|         Pumps|        84491|
|    PL05|        Motors|        84331|
|    PL03|   Controllers|        84202|
|    PL01|        Motors|        84174|
|    PL04|         Pumps|        83976|
|    PL04|        Motors|        83897|
|    PL06|         Pumps|        83884|
|    PL05|         Pumps|        83607|
|    PL05|   Controllers|        83589|
|    PL01|   Controllers|        83569|
+--------+--------------+-------------+
only showing top 20 rows


9. Average temperature by machine.

In [ ]:
summarize("machine_id", F.avg("temperature_c"), "average_temperature").show()

+----------+-------------------+
|machine_id|average_temperature|
+----------+-------------------+
|     MC019| 63.266223444138944|
|     MC020|  63.22489208633087|
|     MC029|  63.13119731057668|
|     MC009|   63.1252779855396|
|     MC014| 63.114309957574186|
|     MC023| 63.109051746112684|
|     MC027| 63.099668282725126|
|     MC022|  63.09403431245128|
|     MC015|  63.09026488395557|
|     MC001|   63.0778733146782|
|     MC017|  63.07674442190667|
|     MC011| 63.060230414746606|
|     MC007|    63.037661716587|
|     MC016|  63.03639870582378|
|     MC026|  63.02672521957329|
|     MC021|  63.01464069491981|
|     MC005| 62.979236446933044|
|     MC028|  62.97779683034558|
|     MC030|  62.96733774497327|
|     MC024|  62.96457946271644|
+----------+-------------------+
only showing top 20 rows


10. Total output by supervisor.

In [ ]:
summarize("supervisor", F.sum("units_produced"), "total_output").show()

+----------+------------+
|supervisor|total_output|
+----------+------------+
|     Farah|    10276986|
|    Harini|    10215803|
|     Deepa|    10130817|
|    Bharat|    10122430|
|     Anita|    10089966|
|    Charan|    10037116|
|     Gopal|    10031531|
|     Irfan|    10023239|
+----------+------------+



**10. Joining DataFrames**

In [ ]:
plant_df = spark.read.csv(
"plant_master.csv",
    header=True,
    inferSchema=True
)
plant_df.show()

+--------+---------------+---------+------+-----------+-------------+
|plant_id|     plant_name|     city|region|year_opened|plant_manager|
+--------+---------------+---------+------+-----------+-------------+
|    PL01|Hyderabad Plant|Hyderabad| South|       2016|     R. Menon|
|    PL02|     Pune Plant|     Pune|  West|       2018|  S. Kulkarni|
|    PL03|  Chennai Plant|  Chennai| South|       2015| A. Natarajan|
|    PL04|    Noida Plant|    Noida| North|       2019|    M. Sharma|
|    PL05|Ahmedabad Plant|Ahmedabad|  West|       2017|     P. Desai|
|    PL06|  Kolkata Plant|  Kolkata|  East|       2020|  D. Banerjee|
+--------+---------------+---------+------+-----------+-------------+



In [ ]:
machine_df = spark.read.csv(
"machine_master.csv",
    header=True,
    inferSchema=True
)
machine_df.show()

+----------+--------+------------+-----------------+--------------+
|machine_id|plant_id|machine_type|capacity_per_hour|installed_year|
+----------+--------+------------+-----------------+--------------+
|     MC001|    PL01|         CNC|              120|          2022|
|     MC002|    PL01|    Assembly|               80|          2024|
|     MC003|    PL01|     Casting|              120|          2018|
|     MC004|    PL01|     Welding|              180|          2024|
|     MC005|    PL01|     Testing|              150|          2022|
|     MC006|    PL02|         CNC|              100|          2021|
|     MC007|    PL02|    Assembly|              100|          2022|
|     MC008|    PL02|     Casting|              180|          2020|
|     MC009|    PL02|     Welding|               80|          2020|
|     MC010|    PL02|     Testing|              150|          2024|
|     MC011|    PL03|         CNC|              100|          2025|
|     MC012|    PL03|    Assembly|              

1. Inner join production with plant master.

In [ ]:
inner_join_df = df.join(plant_df, on="plant_id", how="inner")
inner_join_df.show()

+--------+-------------+---------------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+---------------+---------+------+-----------+-------------+
|plant_id|production_id|production_date|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|     plant_name|     city|region|year_opened|plant_manager|
+--------+-------------+---------------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+---------------+---------+------+-----------+-------------+
|    PL03|    PR0000001|     2026-08-07|     MC011|     OP0275|Evening|     MTR-100|    

2. Display plant name instead of only plant ID.

In [ ]:
df_with_plant_name = df.join(plant_df, on="plant_id", how="inner").select(df["*"], plant_df["plant_name"])
df_with_plant_name.show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+---------------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|     plant_name|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+---------------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Evening|     MTR-100|        Motors|         1060|           993|             30|              34|    388.11|         61.4|          3.88|        Rework| 

3. Join production with machine master.

In [ ]:
production_with_machine_master = df.join(machine_df, on="machine_id", how="inner")
production_with_machine_master.show()

+----------+-------------+---------------+--------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+--------+------------+-----------------+--------------+
|machine_id|production_id|production_date|plant_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|plant_id|machine_type|capacity_per_hour|installed_year|
+----------+-------------+---------------+--------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+--------+------------+-----------------+--------------+
|     MC011|    PR0000001|     2026-08-07|    PL03|     OP0275|Evening|     MTR-100|        Motors| 

4. Create one DataFrame containing production + plant + machine information.

In [ ]:
full_df = df.join(plant_df, on="plant_id", how="inner").join(machine_df, on="machine_id", how="inner")
full_df.show()

+----------+--------+-------------+---------------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+---------------+---------+------+-----------+-------------+--------+------------+-----------------+--------------+
|machine_id|plant_id|production_id|production_date|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|     plant_name|     city|region|year_opened|plant_manager|plant_id|machine_type|capacity_per_hour|installed_year|
+----------+--------+-------------+---------------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+---------------+---------+------+-----

5. Calculate total production by plant city.

In [ ]:
full_df.groupBy("city").agg(sum("units_produced").alias("total_production")).show()

+---------+----------------+
|     city|total_production|
+---------+----------------+
|  Chennai|        13547769|
|Ahmedabad|        13323163|
|  Kolkata|        13492337|
|    Noida|        13529935|
|     Pune|        13546452|
|Hyderabad|        13488232|
+---------+----------------+



6. Calculate average downtime by machine type.

In [ ]:
full_df.groupBy("machine_type").agg(avg("downtime_minutes").alias("average_downtime")).show()

+------------+-----------------+
|machine_type| average_downtime|
+------------+-----------------+
|         CNC|33.64548129466162|
|     Testing|33.87585594724829|
|     Welding|34.00449382099613|
|    Assembly|33.96718284443325|
|     Casting| 34.1660783396419|
+------------+-----------------+



7. Calculate defects by region.

In [ ]:
full_df.groupBy("region").agg(sum("defective_units").alias("total_defects")).show()

+------+-------------+
|region|total_defects|
+------+-------------+
| South|       678984|
|  East|       338993|
|  West|       674149|
| North|       338118|
+------+-------------+



8. Find production records without matching machine master records.


In [ ]:
df.join(machine_df, on="machine_id", how="left_anti").show()

+----------+-------------+---------------+--------+-----------+-----+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|machine_id|production_id|production_date|plant_id|operator_id|shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+----------+-------------+---------------+--------+-----------+-----+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
+----------+-------------+---------------+--------+-----------+-----+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+



9. Perform a left join and inspect unmatched records.

In [ ]:
left_join_df = df.join(machine_df, on="machine_id", how="left")
left_join_df.filter(left_join_df["machine_id"].isNull()).show()

+----------+-------------+---------------+--------+-----------+-----+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+--------+------------+-----------------+--------------+
|machine_id|production_id|production_date|plant_id|operator_id|shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|plant_id|machine_type|capacity_per_hour|installed_year|
+----------+-------------+---------------+--------+-----------+-----+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+--------+------------+-----------------+--------------+
+----------+-------------+---------------+--------+-----------+-----+------------+--------------+---------

10. Find total energy consumption by plant name.

In [ ]:
full_df.groupBy("plant_name").agg(sum("energy_kwh").alias("total_energy_consumption")).show()

+---------------+------------------------+
|     plant_name|total_energy_consumption|
+---------------+------------------------+
|    Noida Plant|       4746238.580000002|
|  Kolkata Plant|       4723180.160000009|
|  Chennai Plant|       4742748.169999995|
|     Pune Plant|       4757984.949999988|
|Hyderabad Plant|       4725471.189999996|
|Ahmedabad Plant|        4672391.03000001|
+---------------+------------------------+



**11. Updating Data**

1. Change Reject to Failed.

In [ ]:
df2 = df2.withColumn("quality_status", F.when(df2["quality_status"] == "Reject", "Failed").otherwise(df2["quality_status"]))
df2.show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+---------------+-------------+--------------+--------------+----------------+----------+---------------------+----------+-------+------------------+-------------------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_consumed|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|production_efficiency|good_units|country| defect_percentage|    energy_per_unit|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+---------------+-------------+--------------+--------------+----------------+----------+---------------------+----------+-------+------------------+-------------------+
|    PR000

2. Change Rework to Requires Review.

In [ ]:
df2 = df2.withColumn("quality_status", F.when(df2["quality_status"] == "Rework", "Requires Review").otherwise(df2["quality_status"]))
df2.show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+---------------+-------------+--------------+---------------+----------------+----------+---------------------+----------+-------+------------------+-------------------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_consumed|temperature_c|vibration_mm_s| quality_status|maintenance_flag|supervisor|production_efficiency|good_units|country| defect_percentage|    energy_per_unit|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+---------------+-------------+--------------+---------------+----------------+----------+---------------------+----------+-------+------------------+-------------------+
|    PR

3. Increase units planned by 5% for Motors.

In [ ]:
df2 = df2.withColumn("units_planned", F.when(df2["product_family"] == "Motors", df2["units_planned"] * 1.05).otherwise(df2["units_planned"]))
df2.show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+------------------+--------------+---------------+----------------+---------------+-------------+--------------+---------------+----------------+----------+---------------------+----------+-------+------------------+-------------------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|     units_planned|units_produced|defective_units|downtime_minutes|energy_consumed|temperature_c|vibration_mm_s| quality_status|maintenance_flag|supervisor|production_efficiency|good_units|country| defect_percentage|    energy_per_unit|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+------------------+--------------+---------------+----------------+---------------+-------------+--------------+---------------+----------------+----------+---------------------+----------+-------+------------------+-------------

4. Increase energy consumption by 3% for Night shifts.

In [ ]:
df2 = df2.withColumn("energy_consumed", F.when(df2["shift"] == "Night", df2["energy_consumed"] * 1.03).otherwise(df2["energy_consumed"]))
df2.show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+------------------+--------------+---------------+----------------+---------------+-------------+--------------+---------------+----------------+----------+---------------------+----------+-------+------------------+-------------------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|     units_planned|units_produced|defective_units|downtime_minutes|energy_consumed|temperature_c|vibration_mm_s| quality_status|maintenance_flag|supervisor|production_efficiency|good_units|country| defect_percentage|    energy_per_unit|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+------------------+--------------+---------------+----------------+---------------+-------------+--------------+---------------+----------------+----------+---------------------+----------+-------+------------------+-------------

5. Replace missing operator IDs with UNASSIGNED.

In [ ]:
df2 = df2.withColumn("operator_id", F.when(df2["operator_id"].isNull(), "UNASSIGNED").otherwise(df2["operator_id"]))
df2.show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+------------------+--------------+---------------+----------------+---------------+-------------+--------------+---------------+----------------+----------+---------------------+----------+-------+------------------+-------------------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|     units_planned|units_produced|defective_units|downtime_minutes|energy_consumed|temperature_c|vibration_mm_s| quality_status|maintenance_flag|supervisor|production_efficiency|good_units|country| defect_percentage|    energy_per_unit|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+------------------+--------------+---------------+----------------+---------------+-------------+--------------+---------------+----------------+----------+---------------------+----------+-------+------------------+-------------

6. Replace supervisor Anita with Anita Rao.

In [ ]:
df2 = df2.withColumn("supervisor", F.when(df2["supervisor"] == "Anita", "Anita Rao").otherwise(df2["supervisor"]))
df2.show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+------------------+--------------+---------------+----------------+---------------+-------------+--------------+---------------+----------------+----------+---------------------+----------+-------+------------------+-------------------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|     units_planned|units_produced|defective_units|downtime_minutes|energy_consumed|temperature_c|vibration_mm_s| quality_status|maintenance_flag|supervisor|production_efficiency|good_units|country| defect_percentage|    energy_per_unit|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+------------------+--------------+---------------+----------------+---------------+-------------+--------------+---------------+----------------+----------+---------------------+----------+-------+------------------+-------------

7. Create a corrected temperature column.

In [ ]:
df2 = df2.withColumn("corrected_temperature", F.when(df2["temperature_c"] > 100, 100).otherwise(df2["temperature_c"]))
df2.show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+------------------+--------------+---------------+----------------+---------------+-------------+--------------+---------------+----------------+----------+---------------------+----------+-------+------------------+-------------------+---------------------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|     units_planned|units_produced|defective_units|downtime_minutes|energy_consumed|temperature_c|vibration_mm_s| quality_status|maintenance_flag|supervisor|production_efficiency|good_units|country| defect_percentage|    energy_per_unit|corrected_temperature|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+------------------+--------------+---------------+----------------+---------------+-------------+--------------+---------------+----------------+----------+---------------------+-------

**12. Null Values and na Functions**

In [ ]:
before = df.count()

1. Count missing operator IDs.

In [ ]:
df.filter(df["operator_id"].isNull()).count()

1524

2. Count missing temperature values.

In [ ]:
df.filter(df["temperature_c"].isNull()).count()

2132

3. Count missing vibration values.

In [ ]:
df.filter(df["vibration_mm_s"].isNull()).count()

1788

4. Count missing downtime.

In [ ]:
df.filter(df["downtime_minutes"].isNull()).count()

1177

5. Use na.drop()

In [ ]:
df.na.drop(subset=["operator_id"]).show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Evening|     MTR-100|        Motors|         1060|           993|             30|              34|    388.11|         61.4|          3.88|        Rework|              No|    Charan|
|    PR0000002|     

6. Drop only rows where temperature is missing.

In [ ]:
df.na.drop(subset=["temperature_c"]).show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Evening|     MTR-100|        Motors|         1060|           993|             30|              34|    388.11|         61.4|          3.88|        Rework|              No|    Charan|
|    PR0000002|     

7. Fill missing downtime with 0.

In [ ]:
df.na.fill({"downtime_minutes": 0}).show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Evening|     MTR-100|        Motors|         1060|           993|             30|              34|    388.11|         61.4|          3.88|        Rework|              No|    Charan|
|    PR0000002|     

8. Fill missing operator ID with UNKNOWN.

In [ ]:
df.na.fill({"operator_id": "UNKNOWN"}).show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Evening|     MTR-100|        Motors|         1060|           993|             30|              34|    388.11|         61.4|          3.88|        Rework|              No|    Charan|
|    PR0000002|     

9. Fill missing temperature with the overall average temperature.

In [ ]:
average_temperature = df.agg(F.avg("temperature_c")).first()[0]
df.na.fill({"temperature_c": average_temperature}).show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Evening|     MTR-100|        Motors|         1060|           993|             30|              34|    388.11|         61.4|          3.88|        Rework|              No|    Charan|
|    PR0000002|     

10. Compare record counts before and after cleansing.

In [ ]:
print("Before cleansing:", before)
print("After cleansing:", df.count())

Before cleansing: 120000
After cleansing: 120000


**13. Built-in Functions**

1. Convert product family to uppercase.

In [ ]:
df2 = df2.withColumn("product_family", F.upper(df2["product_family"]))
df2.show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+------------------+--------------+---------------+----------------+---------------+-------------+--------------+---------------+----------------+----------+---------------------+----------+-------+------------------+-------------------+---------------------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|     units_planned|units_produced|defective_units|downtime_minutes|energy_consumed|temperature_c|vibration_mm_s| quality_status|maintenance_flag|supervisor|production_efficiency|good_units|country| defect_percentage|    energy_per_unit|corrected_temperature|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+------------------+--------------+---------------+----------------+---------------+-------------+--------------+---------------+----------------+----------+---------------------+-------

2. Convert supervisor names to lowercase.

In [ ]:
df2 = df2.withColumn("supervisor", F.lower(df2["supervisor"]))
df2.show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+------------------+--------------+---------------+----------------+---------------+-------------+--------------+---------------+----------------+----------+---------------------+----------+-------+------------------+-------------------+---------------------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|     units_planned|units_produced|defective_units|downtime_minutes|energy_consumed|temperature_c|vibration_mm_s| quality_status|maintenance_flag|supervisor|production_efficiency|good_units|country| defect_percentage|    energy_per_unit|corrected_temperature|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+------------------+--------------+---------------+----------------+---------------+-------------+--------------+---------------+----------------+----------+---------------------+-------

3. Trim operator IDs.

In [ ]:
df2 = df2.withColumn("operator_id", F.trim(df2["operator_id"]))
df2.show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+------------------+--------------+---------------+----------------+---------------+-------------+--------------+---------------+----------------+----------+---------------------+----------+-------+------------------+-------------------+---------------------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|     units_planned|units_produced|defective_units|downtime_minutes|energy_consumed|temperature_c|vibration_mm_s| quality_status|maintenance_flag|supervisor|production_efficiency|good_units|country| defect_percentage|    energy_per_unit|corrected_temperature|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+------------------+--------------+---------------+----------------+---------------+-------------+--------------+---------------+----------------+----------+---------------------+-------

4. Find length of product codes.

In [ ]:
df2 = df2.withColumn("product_code_length", F.length(df2["product_code"]))
df2.show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+------------------+--------------+---------------+----------------+---------------+-------------+--------------+---------------+----------------+----------+---------------------+----------+-------+------------------+-------------------+---------------------+-------------------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|     units_planned|units_produced|defective_units|downtime_minutes|energy_consumed|temperature_c|vibration_mm_s| quality_status|maintenance_flag|supervisor|production_efficiency|good_units|country| defect_percentage|    energy_per_unit|corrected_temperature|product_code_length|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+------------------+--------------+---------------+----------------+---------------+-------------+--------------+---------------+----------------+

5. Combine plant ID and machine ID.

In [ ]:
df2 = df2.withColumn("plant_machine_id", F.concat(df2["plant_id"], F.lit("-"), df2["machine_id"]))
df2.show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+------------------+--------------+---------------+----------------+---------------+-------------+--------------+---------------+----------------+----------+---------------------+----------+-------+------------------+-------------------+---------------------+-------------------+----------------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|     units_planned|units_produced|defective_units|downtime_minutes|energy_consumed|temperature_c|vibration_mm_s| quality_status|maintenance_flag|supervisor|production_efficiency|good_units|country| defect_percentage|    energy_per_unit|corrected_temperature|product_code_length|plant_machine_id|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+------------------+--------------+---------------+----------------+---------------+-------------+--------------

6. Round temperature to one decimal.

In [ ]:
df2 = df2.withColumn("rounded_temperature", F.round(df2["temperature_c"], 1))
df2.show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+------------------+--------------+---------------+----------------+---------------+-------------+--------------+---------------+----------------+----------+---------------------+----------+-------+------------------+-------------------+---------------------+-------------------+----------------+-------------------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|     units_planned|units_produced|defective_units|downtime_minutes|energy_consumed|temperature_c|vibration_mm_s| quality_status|maintenance_flag|supervisor|production_efficiency|good_units|country| defect_percentage|    energy_per_unit|corrected_temperature|product_code_length|plant_machine_id|rounded_temperature|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+------------------+--------------+---------------+----------------+----

7. Calculate absolute difference between planned and produced units.

In [ ]:
df2 = df2.withColumn("unit_difference", F.abs(df2["units_planned"] - df2["units_produced"]))
df2.show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+------------------+--------------+---------------+----------------+---------------+-------------+--------------+---------------+----------------+----------+---------------------+----------+-------+------------------+-------------------+---------------------+-------------------+----------------+-------------------+-----------------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|     units_planned|units_produced|defective_units|downtime_minutes|energy_consumed|temperature_c|vibration_mm_s| quality_status|maintenance_flag|supervisor|production_efficiency|good_units|country| defect_percentage|    energy_per_unit|corrected_temperature|product_code_length|plant_machine_id|rounded_temperature|  unit_difference|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+------------------+--------------+-

8. Replace characters in production ID using regexp_replace.

In [ ]:
df2 = df2.withColumn("production_id", F.regexp_replace(df2["production_id"], "-", ""))
df2.show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+------------------+--------------+---------------+----------------+---------------+-------------+--------------+---------------+----------------+----------+---------------------+----------+-------+------------------+-------------------+---------------------+-------------------+----------------+-------------------+-----------------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|     units_planned|units_produced|defective_units|downtime_minutes|energy_consumed|temperature_c|vibration_mm_s| quality_status|maintenance_flag|supervisor|production_efficiency|good_units|country| defect_percentage|    energy_per_unit|corrected_temperature|product_code_length|plant_machine_id|rounded_temperature|  unit_difference|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+------------------+--------------+-

9. Extract part of product code using substring.

In [ ]:
df2 = df2.withColumn("product_code_part", F.substring(df2["product_code"], 1, 3))
df2.show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+------------------+--------------+---------------+----------------+---------------+-------------+--------------+---------------+----------------+----------+---------------------+----------+-------+------------------+-------------------+---------------------+-------------------+----------------+-------------------+-----------------+-----------------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|     units_planned|units_produced|defective_units|downtime_minutes|energy_consumed|temperature_c|vibration_mm_s| quality_status|maintenance_flag|supervisor|production_efficiency|good_units|country| defect_percentage|    energy_per_unit|corrected_temperature|product_code_length|plant_machine_id|rounded_temperature|  unit_difference|product_code_part|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------

10. Use coalesce() to return temperature or a default value.

In [ ]:
df2 = df2.withColumn("corrected_temperature", F.coalesce(df2["temperature_c"], F.lit(25.0)))
df2.show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+------------------+--------------+---------------+----------------+---------------+-------------+--------------+---------------+----------------+----------+---------------------+----------+-------+------------------+-------------------+---------------------+-------------------+----------------+-------------------+-----------------+-----------------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|     units_planned|units_produced|defective_units|downtime_minutes|energy_consumed|temperature_c|vibration_mm_s| quality_status|maintenance_flag|supervisor|production_efficiency|good_units|country| defect_percentage|    energy_per_unit|corrected_temperature|product_code_length|plant_machine_id|rounded_temperature|  unit_difference|product_code_part|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------

**14. when() / otherwise()**

Create:

**Efficiency Category**

gt= 100%  → Above Target
gt= 95%   → On Target
gt= 85%   → Below Target
otherwise → Critical

**Defect Category**

0               → Perfect
1–10            → Normal
11–25           → Attention
above 25        → High Defect

**Machine Condition**

Use temperature and vibration:
High temperature OR high vibration → Inspect
otherwise                          → Normal

Then count records in each resulting category.

In [ ]:
from pyspark.sql import functions as F

# Calculate production efficiency if not already present
df_categorized = df.withColumn(
    "production_efficiency",
    (F.col("units_produced") / F.col("units_planned")) * 100
)

# 1. Create Efficiency Category
df_categorized = df_categorized.withColumn(
    "efficiency_category",
    F.when(F.col("production_efficiency") >= 100, "Above Target")
     .when(F.col("production_efficiency") >= 95, "On Target")
     .when(F.col("production_efficiency") >= 85, "Below Target")
     .otherwise("Critical")
)

# 2. Create Defect Category
df_categorized = df_categorized.withColumn(
    "defect_category",
    F.when(F.col("defective_units") == 0, "Perfect")
     .when(F.col("defective_units") <= 10, "Normal")
     .when(F.col("defective_units") <= 25, "Attention")
     .otherwise("High Defect")
)

# 3. Create Machine Condition Category
# Use overall averages as reference for threshold if desired, or define specific thresholds.
# Here we define high temperature as > 75 (reference to earlier cells) or vibration > 4.5.
# Adjusting thresholds safely to make sure both conditions function properly.
temp_threshold = 75.0
vib_threshold = 4.5

df_categorized = df_categorized.withColumn(
    "machine_condition",
    F.when(
        (F.col("temperature_c") > temp_threshold) | (F.col("vibration_mm_s") > vib_threshold),
        "Inspect"
    ).otherwise("Normal")
)

# Display some sample output with the new categories
df_categorized.select(
    "production_id",
    "production_efficiency",
    "efficiency_category",
    "defective_units",
    "defect_category",
    "temperature_c",
    "vibration_mm_s",
    "machine_condition"
).show(10)

# Count records in each category
print("=== Efficiency Category Counts ===")
df_categorized.groupBy("efficiency_category").count().show()

print("=== Defect Category Counts ===")
df_categorized.groupBy("defect_category").count().show()

print("=== Machine Condition Counts ===")
df_categorized.groupBy("machine_condition").count().show()

+-------------+---------------------+-------------------+---------------+---------------+-------------+--------------+-----------------+
|production_id|production_efficiency|efficiency_category|defective_units|defect_category|temperature_c|vibration_mm_s|machine_condition|
+-------------+---------------------+-------------------+---------------+---------------+-------------+--------------+-----------------+
|    PR0000001|    93.67924528301886|       Below Target|             30|    High Defect|         61.4|          3.88|           Normal|
|    PR0000002|   103.33745364647713|       Above Target|              0|        Perfect|        64.37|          2.28|           Normal|
|    PR0000003|    85.97285067873304|       Below Target|             17|      Attention|        54.13|          4.66|          Inspect|
|    PR0000004|     91.4866581956798|       Below Target|              7|         Normal|        64.47|          NULL|           Normal|
|    PR0000005|    86.04651162790698|    

**15. Complex JSON**

In [ ]:
machine_json = spark.read.option(
"multiline", "true"
).json("machine_configurations.json")

1. Display schema.

In [ ]:
machine_json.printSchema()

root
 |-- capacity_per_hour: long (nullable = true)
 |-- installed_year: long (nullable = true)
 |-- machine_id: string (nullable = true)
 |-- machine_type: string (nullable = true)
 |-- maintenance: struct (nullable = true)
 |    |-- frequency_days: long (nullable = true)
 |    |-- last_service: string (nullable = true)
 |-- plant_id: string (nullable = true)
 |-- sensors: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- active: boolean (nullable = true)
 |    |    |-- sensor: string (nullable = true)
 |    |    |-- unit: string (nullable = true)
 |-- specifications: struct (nullable = true)
 |    |-- max_temperature_c: long (nullable = true)
 |    |-- rated_power_kw: long (nullable = true)
 |    |-- vibration_limit_mm_s: double (nullable = true)



2. Display machine ID and machine type.

In [ ]:
machine_json.select("machine_id", "machine_type").show()

+----------+------------+
|machine_id|machine_type|
+----------+------------+
|     MC001|         CNC|
|     MC002|    Assembly|
|     MC003|     Casting|
|     MC004|     Welding|
|     MC005|     Testing|
|     MC006|         CNC|
|     MC007|    Assembly|
|     MC008|     Casting|
|     MC009|     Welding|
|     MC010|     Testing|
|     MC011|         CNC|
|     MC012|    Assembly|
|     MC013|     Casting|
|     MC014|     Welding|
|     MC015|     Testing|
|     MC016|         CNC|
|     MC017|    Assembly|
|     MC018|     Casting|
|     MC019|     Welding|
|     MC020|     Testing|
+----------+------------+
only showing top 20 rows


3. Extract specifications.rated_power_kw.

In [ ]:
machine_json.select("machine_id", "specifications.rated_power_kw").show()

+----------+--------------+
|machine_id|rated_power_kw|
+----------+--------------+
|     MC001|            22|
|     MC002|            22|
|     MC003|            55|
|     MC004|            22|
|     MC005|            30|
|     MC006|            30|
|     MC007|            22|
|     MC008|            55|
|     MC009|            30|
|     MC010|            55|
|     MC011|            30|
|     MC012|            55|
|     MC013|            30|
|     MC014|            30|
|     MC015|            45|
|     MC016|            30|
|     MC017|            30|
|     MC018|            30|
|     MC019|            30|
|     MC020|            15|
+----------+--------------+
only showing top 20 rows


4. Extract maximum temperature.

In [ ]:
machine_json.select("machine_id", "specifications.max_temperature_c").show()

+----------+-----------------+
|machine_id|max_temperature_c|
+----------+-----------------+
|     MC001|               85|
|     MC002|               85|
|     MC003|               90|
|     MC004|               85|
|     MC005|               85|
|     MC006|               85|
|     MC007|               80|
|     MC008|               75|
|     MC009|               85|
|     MC010|               90|
|     MC011|               90|
|     MC012|               80|
|     MC013|               90|
|     MC014|               80|
|     MC015|               90|
|     MC016|               75|
|     MC017|               85|
|     MC018|               85|
|     MC019|               75|
|     MC020|               80|
+----------+-----------------+
only showing top 20 rows


5. Extract vibration limit.

In [ ]:
machine_json.select("machine_id", "specifications.vibration_limit_mm_s").show()

+----------+--------------------+
|machine_id|vibration_limit_mm_s|
+----------+--------------------+
|     MC001|                 5.0|
|     MC002|                 5.5|
|     MC003|                 4.5|
|     MC004|                 4.5|
|     MC005|                 4.5|
|     MC006|                 5.5|
|     MC007|                 4.5|
|     MC008|                 5.0|
|     MC009|                 5.5|
|     MC010|                 4.5|
|     MC011|                 6.0|
|     MC012|                 5.5|
|     MC013|                 5.0|
|     MC014|                 5.5|
|     MC015|                 4.5|
|     MC016|                 5.5|
|     MC017|                 4.5|
|     MC018|                 5.0|
|     MC019|                 5.0|
|     MC020|                 5.0|
+----------+--------------------+
only showing top 20 rows


6. Display maintenance frequency.

In [ ]:
machine_json.select("machine_id", "maintenance.frequency_days").show()

+----------+--------------+
|machine_id|frequency_days|
+----------+--------------+
|     MC001|            60|
|     MC002|            60|
|     MC003|            45|
|     MC004|            60|
|     MC005|            30|
|     MC006|            60|
|     MC007|            30|
|     MC008|            30|
|     MC009|            30|
|     MC010|            45|
|     MC011|            30|
|     MC012|            30|
|     MC013|            60|
|     MC014|            90|
|     MC015|            30|
|     MC016|            45|
|     MC017|            30|
|     MC018|            30|
|     MC019|            30|
|     MC020|            45|
+----------+--------------+
only showing top 20 rows


7. Work with the sensors array.

In [ ]:
machine_json.select("machine_id", "sensors").show()

+----------+--------------------+
|machine_id|             sensors|
+----------+--------------------+
|     MC001|[{true, temperatu...|
|     MC002|[{true, temperatu...|
|     MC003|[{true, temperatu...|
|     MC004|[{true, temperatu...|
|     MC005|[{true, temperatu...|
|     MC006|[{true, temperatu...|
|     MC007|[{true, temperatu...|
|     MC008|[{true, temperatu...|
|     MC009|[{true, temperatu...|
|     MC010|[{true, temperatu...|
|     MC011|[{true, temperatu...|
|     MC012|[{true, temperatu...|
|     MC013|[{true, temperatu...|
|     MC014|[{true, temperatu...|
|     MC015|[{true, temperatu...|
|     MC016|[{true, temperatu...|
|     MC017|[{true, temperatu...|
|     MC018|[{true, temperatu...|
|     MC019|[{true, temperatu...|
|     MC020|[{true, temperatu...|
+----------+--------------------+
only showing top 20 rows


8. Explode the sensors array.

In [ ]:
exploded_sensors = machine_json.select(
    "machine_id",
    F.explode("sensors").alias("sensor")
)
exploded_sensors.show()

+----------+--------------------+
|machine_id|              sensor|
+----------+--------------------+
|     MC001|{true, temperatur...|
|     MC001|{true, vibration,...|
|     MC001| {true, energy, kWh}|
|     MC002|{true, temperatur...|
|     MC002|{true, vibration,...|
|     MC002| {true, energy, kWh}|
|     MC003|{true, temperatur...|
|     MC003|{true, vibration,...|
|     MC003| {true, energy, kWh}|
|     MC004|{true, temperatur...|
|     MC004|{true, vibration,...|
|     MC004| {true, energy, kWh}|
|     MC005|{true, temperatur...|
|     MC005|{true, vibration,...|
|     MC005| {true, energy, kWh}|
|     MC006|{true, temperatur...|
|     MC006|{true, vibration,...|
|     MC006| {true, energy, kWh}|
|     MC007|{true, temperatur...|
|     MC007|{true, vibration,...|
+----------+--------------------+
only showing top 20 rows


9. Display one row per machine/sensor combination.

In [ ]:
exploded_sensors.select("machine_id", "sensor.*").show()

+----------+------+-----------+----+
|machine_id|active|     sensor|unit|
+----------+------+-----------+----+
|     MC001|  true|temperature|   C|
|     MC001|  true|  vibration|mm/s|
|     MC001|  true|     energy| kWh|
|     MC002|  true|temperature|   C|
|     MC002|  true|  vibration|mm/s|
|     MC002|  true|     energy| kWh|
|     MC003|  true|temperature|   C|
|     MC003|  true|  vibration|mm/s|
|     MC003|  true|     energy| kWh|
|     MC004|  true|temperature|   C|
|     MC004|  true|  vibration|mm/s|
|     MC004|  true|     energy| kWh|
|     MC005|  true|temperature|   C|
|     MC005|  true|  vibration|mm/s|
|     MC005|  true|     energy| kWh|
|     MC006|  true|temperature|   C|
|     MC006|  true|  vibration|mm/s|
|     MC006|  true|     energy| kWh|
|     MC007|  true|temperature|   C|
|     MC007|  true|  vibration|mm/s|
+----------+------+-----------+----+
only showing top 20 rows


10. Find machines having an active temperature sensor.

In [ ]:
exploded_sensors.filter(exploded_sensors["sensor.sensor"] == "temperature").select("machine_id").distinct().show()

+----------+
|machine_id|
+----------+
|     MC003|
|     MC018|
|     MC017|
|     MC009|
|     MC014|
|     MC007|
|     MC002|
|     MC024|
|     MC016|
|     MC026|
|     MC005|
|     MC010|
|     MC012|
|     MC023|
|     MC021|
|     MC020|
|     MC011|
|     MC013|
|     MC025|
|     MC028|
+----------+
only showing top 20 rows


11. Join machine configuration JSON with production data.

In [ ]:
joined_config_df = df.join(machine_json, on="machine_id", how="inner")
joined_config_df.show(5)

+----------+-------------+---------------+--------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+-----------------+--------------+------------+----------------+--------+--------------------+--------------+
|machine_id|production_id|production_date|plant_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|capacity_per_hour|installed_year|machine_type|     maintenance|plant_id|             sensors|specifications|
+----------+-------------+---------------+--------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+-----------------+--------------+------------+----

**16. Union and Union ALL**

In [ ]:
extra_df = spark.read.csv(
"manufacturing_production_extra_12k.csv",
    header=True,
    inferSchema=True
)

1. Count records in the main DataFrame.

In [ ]:
print(df.count())

120000


2. Count records in the extra DataFrame.

In [ ]:
print(extra_df.count())

12500


3. Combine using union().

In [ ]:
u1 = df.union(extra_df)
u1.show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Evening|     MTR-100|        Motors|         1060|           993|             30|              34|    388.11|         61.4|          3.88|        Rework|              No|    Charan|
|    PR0000002|     

4. Combine using unionAll().

In [ ]:
u2 = df.unionAll(extra_df)
u2.show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Evening|     MTR-100|        Motors|         1060|           993|             30|              34|    388.11|         61.4|          3.88|        Rework|              No|    Charan|
|    PR0000002|     

5. Compare resulting counts.

In [ ]:
print(u1.count())
print(u2.count())

132500
132500


6. Identify duplicate production_id values.

In [ ]:
u1.groupBy("production_id").count().filter("count > 1").show()

+-------------+-----+
|production_id|count|
+-------------+-----+
|    PR0000413|    2|
|    PR0000135|    2|
|    PR0000236|    2|
|    PR0000339|    2|
|    PR0000059|    2|
|    PR0000463|    2|
|    PR0000136|    2|
|    PR0000263|    2|
|    PR0000393|    2|
|    PR0000445|    2|
|    PR0000124|    2|
|    PR0000299|    2|
|    PR0000341|    2|
|    PR0000008|    2|
|    PR0000172|    2|
|    PR0000359|    2|
|    PR0000389|    2|
|    PR0000026|    2|
|    PR0000274|    2|
|    PR0000161|    2|
+-------------+-----+
only showing top 20 rows


7. Remove duplicates after combining.

In [ ]:
delete_duplicates = u1.dropDuplicates(["production_id"])
delete_duplicates.show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0000002|     2026-08-03|    PL04|     MC019|     OP0489|Morning|     CTL-330|   Controllers|          809|           836|              0|               0|    398.58|        64.37|          2.28|        Reject|              No|     Farah|
|    PR0000003|     

8. Compare unioned count before and after dropDuplicates().

In [ ]:
print("Before:", u1.count())
print("After:", delete_duplicates.count())

Before: 132500
After: 132000


9. Try unionByName().

In [ ]:
u3 = df.unionByName(extra_df)
u3.show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Evening|     MTR-100|        Motors|         1060|           993|             30|              34|    388.11|         61.4|          3.88|        Rework|              No|    Charan|
|    PR0000002|     

10. Rearrange columns in one DataFrame and use unionByName() appropriately.

In [ ]:
reordered = extra_df.select(*reversed(extra_df.columns))
u4 = df.unionByName(reordered)
print(u4.count())

132500


**17. Window Functions**


In [ ]:
from pyspark.sql.window import Window

In [ ]:
w_plant_units = Window.partitionBy("plant_id").orderBy("units_produced")

1. Rank production records within each plant by units produced.

In [ ]:
df2 = df.withColumn("rank", F.rank().over(w_plant_units))
df2.show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+----+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|rank|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+----+
|    PR0038643|     2026-08-06|    PL02|     MC009|     OP0361|Evening|     MTR-100|        Motors|          250|           205|              6|              73|     65.18|        55.99|          2.72|          Pass|              No|    Harini|   1|


2. Use row_number().

In [ ]:
df2 = df.withColumn("row_number", F.row_number().over(w_plant_units))
df2.show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|row_number|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+----------+
|    PR0038643|     2026-08-06|    PL02|     MC009|     OP0361|Evening|     MTR-100|        Motors|          250|           205|              6|              73|     65.18|        55.99|          2.72|          Pass|              No

3. Use rank().

In [ ]:
df2 = df.withColumn("rank", F.rank().over(w_plant_units))
df2.show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+----+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|rank|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+----+
|    PR0038643|     2026-08-06|    PL02|     MC009|     OP0361|Evening|     MTR-100|        Motors|          250|           205|              6|              73|     65.18|        55.99|          2.72|          Pass|              No|    Harini|   1|


4. Use dense_rank().

In [ ]:
df2 = df.withColumn("dense_rank", F.dense_rank().over(w_plant_units))
df2.show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|dense_rank|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+----------+
|    PR0038643|     2026-08-06|    PL02|     MC009|     OP0361|Evening|     MTR-100|        Motors|          250|           205|              6|              73|     65.18|        55.99|          2.72|          Pass|              No

5. Find top 5 production runs per plant.

In [ ]:
df2 = df.withColumn("rank", F.rank().over(w_plant_units))
df2.filter(df2["rank"] <= 5).show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+----+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|rank|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+----+
|    PR0030899|     2026-05-24|    PL01|     MC004|     OP0380|Evening|     VLV-120|        Valves|          252|           206|             12|              12|     75.94|        50.77|          3.34|        Rework|              No|     Farah|   1|


6. Find top 3 machines by output within each plant.

In [ ]:
machine_totals = Window.partitionBy("plant_id").orderBy(F.desc("units_produced"))
df2 = df.withColumn("rank", F.rank().over(machine_totals))
df2.filter(df2["rank"] <= 3).show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+----+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|rank|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+----+
|    PR0073059|     2026-04-01|    PL01|     MC003|     OP0110|Evening|     MTR-200|        Motors|         1198|          1239|             11|               0|     282.6|        50.32|          3.91|        Rework|              No|     Gopal|   1|


7. Calculate total plant production without collapsing rows.

In [ ]:
df2 = df.withColumn("total_production", F.sum("units_produced").over(w_plant_units))
df2.show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+----------------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|total_production|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+----------------+
|    PR0038643|     2026-08-06|    PL02|     MC009|     OP0361|Evening|     MTR-100|        Motors|          250|           205|              6|              73|     65.18|        55.99|          2.72|          Pas

8. Calculate average machine output using a window.

In [ ]:
w_machine = Window.partitionBy("machine_id")
df2 = df.withColumn("avg_output", F.avg("units_produced").over(w_machine))
df2.show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+-----------------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|       avg_output|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+-----------------+
|    PR0000003|     2026-05-20|    PL01|     MC002|     OP0189|Evening|     MTR-300|        Motors|          442|           380|             17|              15|    183.34|        54.13|          4.66|        Re

9. Calculate cumulative production by machine ordered by date.

In [ ]:
df2 = df.withColumn("cumulative_production", F.sum("units_produced").over(Window.partitionBy("machine_id").orderBy("production_date")))
df2.show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+---------------------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|cumulative_production|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+---------------------+
|    PR0001004|     2026-01-01|    PL01|     MC002|     OP0424|  Night|     VLV-120|        Valves|          517|           520|             14|               0|    133.77|        52.65|          4.5

10. Display previous production output using lag().

In [ ]:
df2 = df.withColumn("previous_output", F.lag("units_produced").over(Window.partitionBy("machine_id").orderBy("production_date")))
df2.show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+---------------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|previous_output|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+---------------+
|    PR0001004|     2026-01-01|    PL01|     MC002|     OP0424|  Night|     VLV-120|        Valves|          517|           520|             14|               0|    133.77|        52.65|          4.53|          Pass| 

11. Display next production output using lead().

In [ ]:
df2 = df.withColumn("next_output", F.lead("units_produced").over(Window.partitionBy("machine_id").orderBy("production_date")))
df2.show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+-----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|next_output|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+-----------+
|    PR0001004|     2026-01-01|    PL01|     MC002|     OP0424|  Night|     VLV-120|        Valves|          517|           520|             14|               0|    133.77|        52.65|          4.53|          Pass|             

12. Calculate change from the previous production run.

In [ ]:
df2 = df.withColumn("change_from_previous_output", F.col("units_produced") - F.lag("units_produced").over(Window.partitionBy("machine_id").orderBy("production_date")))
df2.show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+---------------------------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|change_from_previous_output|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+---------------------------+
|    PR0001004|     2026-01-01|    PL01|     MC002|     OP0424|  Night|     VLV-120|        Valves|          517|           520|             14|               0|    133.77|        5

13. Rank product families by total output.

In [ ]:
df2 = df.groupBy("product_family").agg(F.sum("units_produced").alias("total_output"))
df2 = df2.withColumn("rank", F.rank().over(Window.orderBy(F.desc("total_output"))))
df2.show()

+--------------+------------+----+
|product_family|total_output|rank|
+--------------+------------+----+
|        Valves|    20433082|   1|
|         Pumps|    20208739|   2|
|   Controllers|    20168572|   3|
|        Motors|    20117495|   4|
+--------------+------------+----+



**18. Date and Time Functions**

In [ ]:
from pyspark.sql.functions import to_date, year, month, day

Using production_date:

1. Convert to DateType.
2. Extract year.
3. Extract month.
4. Extract day.
5. Extract month name.
6. Extract day name.
7. Create year_month .
8. Calculate monthly production.
9. Calculate monthly defects.
10. Calculate production by month and plant.
15. Find the month with the highest total output.

In [ ]:
# Base DataFrame with date parts (never overwrite this one)
df_dates = (df
    .withColumn("production_date", F.to_date("production_date")) #1
    .withColumn("year", F.year("production_date")) #2
    .withColumn("month", F.month("production_date")) #3
    .withColumn("day", F.dayofmonth("production_date")) #4
    .withColumn("month_name", F.date_format("production_date", "MMMM")) #5
    .withColumn("day_name", F.date_format("production_date", "EEEE")) #6
    .withColumn("year_month", F.date_format("production_date", "yyyy-MM"))) #7

# 8. Monthly production
monthly_prod = (df_dates.groupBy("year_month")
    .agg(F.sum("units_produced").alias("monthly_production"))
    .orderBy("year_month"))
monthly_prod.show()

# 9. Monthly defects
monthly_def = (df_dates.groupBy("year_month")
    .agg(F.sum("defective_units").alias("monthly_defects"))
    .orderBy("year_month"))
monthly_def.show()

# 10. Production by month and plant
(df_dates.groupBy("year_month", "plant_id")
    .agg(F.sum("units_produced").alias("total_production"))
    .orderBy("year_month", "plant_id").show())

# 15. Month with the highest total output
monthly_prod.orderBy(F.desc("monthly_production")).limit(1).show()

+----------+------------------+
|year_month|monthly_production|
+----------+------------------+
|   2026-01|           8478059|
|   2026-02|           7810891|
|   2026-03|           8441121|
|   2026-04|           8211005|
|   2026-05|           8507542|
|   2026-06|           8400505|
|   2026-07|           8476416|
|   2026-08|           8529150|
|   2026-09|           8115221|
|   2026-10|           5957978|
+----------+------------------+

+----------+---------------+
|year_month|monthly_defects|
+----------+---------------+
|   2026-01|         211741|
|   2026-02|         197462|
|   2026-03|         212036|
|   2026-04|         208707|
|   2026-05|         212049|
|   2026-06|         210381|
|   2026-07|         210482|
|   2026-08|         214807|
|   2026-09|         202990|
|   2026-10|         149589|
+----------+---------------+

+----------+--------+----------------+
|year_month|plant_id|total_production|
+----------+--------+----------------+
|   2026-01|    PL01|      

11. Find production between two dates.

In [ ]:
df_dt = df_dates.filter(
    (df_dates["production_date"] >= "2023-01-01") &
    (df_dates["production_date"] <= "2023-12-31")
)
df_dt.show()

+-------------+---------------+--------+----------+-----------+-----+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+----+-----+---+----------+--------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|year|month|day|month_name|day_name|year_month|
+-------------+---------------+--------+----------+-----------+-----+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+----+-----+---+----------+--------+----------+
+-------------+---------------+--------+----------+-----------+-----+------------+--------------+-------------+--------------+-------

12. Add 7 days to production date.

In [ ]:
df_dt = df_dates.withColumn("plus_7_date", F.date_add("production_date", 7))
df_dt.show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+----+-----+---+----------+---------+----------+-----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|year|month|day|month_name| day_name|year_month|plus_7_date|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+----+-----+---+----------+---------+----------+-----------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Evening|     MTR-100|    

13. Subtract 30 days.

In [ ]:
df_dt = df_dates.withColumn("minus_30_days", F.date_sub("production_date", 30))
df_dt.show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+----+-----+---+----------+---------+----------+-------------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|year|month|day|month_name| day_name|year_month|minus_30_days|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+----+-----+---+----------+---------+----------+-------------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Evening|     MTR-10

14. Calculate difference from current date.

In [ ]:
df_dt = df_dates.withColumn("days_from_today", F.datediff(F.current_date(), "production_date"))
df_dt.show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+----+-----+---+----------+---------+----------+---------------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|year|month|day|month_name| day_name|year_month|days_from_today|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+----+-----+---+----------+---------+----------+---------------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Evening|     